In [304]:
import pandas as pd
import numpy as np

In [305]:
# Load dataset
df = pd.read_csv("dirty_cafe_sales.csv")

## 1 Checking Basic information

In [306]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [307]:
rows,columns = df.shape
print("Rows:",rows)
print("Columns:",columns)
print("Column names:",df.columns.tolist())

Rows: 10000
Columns: 8
Column names: ['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']


In [308]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [309]:
df.isna().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

## 2 Standerize dataset

### 1 Item column

In [310]:
# count item names where is error
df['Item'][df['Item'] == "ERROR"].count()

np.int64(292)

In [311]:
# replace ERROR to NaN
df.loc[df['Item'] == 'ERROR'] = np.nan

In [312]:
df['Item'][df['Item'] == "ERROR"].count()

np.int64(0)

In [313]:
# Keep for now
df['Item'][df['Item'] == "UNKNOWN"].count()

np.int64(344)

In [314]:
# stander item name strip and title
def strip_and_title(col:str):
    if not col in df.columns.tolist():
        raise KeyError('UNKNOWN: column name not exists')
    df[col] = df[col].astype(str)
    df[col] = df[col].str.strip()
    df[col] = df[col].str.title()

strip_and_title('Item')
df['Item']

0         Coffee
1           Cake
2         Cookie
3          Salad
4         Coffee
          ...   
9995      Coffee
9996         NaN
9997      Coffee
9998      Cookie
9999    Sandwich
Name: Item, Length: 10000, dtype: str

In [315]:
df['Item'].unique()

<StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',  'Unknown',
 'Sandwich',        nan,    'Juice',      'Tea']
Length: 10, dtype: str

### 2 Quantity column

In [316]:
df['Quantity']

0       2
1       4
2       4
3       2
4       2
       ..
9995    2
9996    3
9997    4
9998    3
9999    3
Name: Quantity, Length: 10000, dtype: str

In [317]:
# Count unknown and error
df['Quantity'][(df['Quantity'] == "ERROR") | (df['Quantity'] == "UNKNOWN")].count()

np.int64(328)

In [318]:
# replace to nan where unknown or error
df.loc[(df['Quantity'] == "ERROR") | (df['Quantity'] == "UNKNOWN")] = np.nan

In [319]:
df['Quantity'][(df['Quantity'] == "ERROR") | (df['Quantity'] == "UNKNOWN")].count()

np.int64(0)

In [320]:
# type casting str to int
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce').astype('Int64')
df['Quantity']

0       2
1       4
2       4
3       2
4       2
       ..
9995    2
9996    3
9997    4
9998    3
9999    3
Name: Quantity, Length: 10000, dtype: Int64

### 3 Pricing columns price per unit and total spent

In [321]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [322]:
df[['Price Per Unit','Total Spent']][df[['Price Per Unit','Total Spent']].isin(['ERROR','UNKNOWN'])].count()

Price Per Unit    341
Total Spent       309
dtype: int64

In [323]:
df.replace({
    'Price Per Unit': {'ERROR': np.nan, 'UNKNOWN': np.nan },
    'Total Spent': {'ERROR': np.nan, 'UNKNOWN': np.nan }
}, inplace=True)

df


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [324]:
df[['Price Per Unit','Total Spent']][df[['Price Per Unit','Total Spent']].isin(['ERROR','UNKNOWN'])].count()

Price Per Unit    0
Total Spent       0
dtype: int64

In [325]:
# to numeric
def convert_to_numeric(col):
    if not col in df.columns.tolist():
        raise KeyError("UNKNOWN: column does not exists")

    df[col] = pd.to_numeric(df[col],errors="coerce")

convert_to_numeric('Price Per Unit')
convert_to_numeric('Total Spent')
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


### 4 Location,Payment Method and Transaction Date Columns

In [326]:
df[['Location','Transaction Date','Payment Method']][df[['Location','Transaction Date','Payment Method']].isin(['ERROR','UNKNOWN'])].count()

Location            652
Transaction Date    283
Payment Method      565
dtype: int64

In [327]:
df.replace({
    'Location':{'ERROR':np.nan,'UNKNOWN':np.nan},
    'Transaction Date':{'ERROR':np.nan,'UNKNOWN':np.nan},
    'Payment Method':{'ERROR':np.nan,'UNKNOWN':np.nan},

},inplace=True)

df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,NaN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [328]:
df[['Location','Transaction Date','Payment Method']][df[['Location','Transaction Date','Payment Method']].isin(['ERROR','UNKNOWN'])].count()

Location            0
Transaction Date    0
Payment Method      0
dtype: int64

In [329]:
strip_and_title('Location')
strip_and_title('Payment Method')

df['Transaction Date'] = pd.to_datetime(df['Transaction Date'],errors="coerce")

df['Payment Method'] = df['Payment Method'].astype('category')
df['Location'] = df['Location'].astype('category')

df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-Store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-Store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-Store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,NaN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [337]:
df['Payment Method'].unique()

['Credit Card', 'Cash', NaN, 'Digital Wallet']
Categories (3, str): ['Cash', 'Credit Card', 'Digital Wallet']

In [338]:
df['Location'].unique()

['Takeaway', 'In-Store', NaN]
Categories (2, str): ['In-Store', 'Takeaway']

In [339]:
df['Transaction Date'].info()

<class 'pandas.Series'>
RangeIndex: 10000 entries, 0 to 9999
Series name: Transaction Date
Non-Null Count  Dtype         
--------------  -----         
8950 non-null   datetime64[us]
dtypes: datetime64[us](1)
memory usage: 78.3 KB
